# The baseline noise (`bstd`) of each channel

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

First step of a pulse-quality cleanup: the distribution of `bstd`, the standard deviation of the baseline of a trace (`baseline_std` of `goldsluice`'s `baseline`: the standard deviation of samples 10 to 500, in ADC counts), for each of the six channels. A trace with a quiet baseline has a `bstd` at the noise of the channel; a trace with a pulse before the trigger, a drifting baseline or a glitch in the window has a larger one, so the upper tail is what a cut would remove.

**Data.** The eight DCRC1pB-triggered series near -82 V of `r76-82v-dcrc1pb.dataset.toml` (output `~/goldsluice-output/r76-82v-dcrc1pb`). Real triggers (`Unknown`) are drawn in blue and random triggers (`Physics`) in grey, normalised separately; the random triggers have no pulse, so they show the noise of the channel by itself. The series are also drawn one by one, to see whether the noise changes with time. Run with the `darkmatter_cli_env` kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from goldpan.dataset import Dataset

LETTERS = "ABCDEF"
_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(PROCESSING / "r76-82v-dcrc1pb.dataset.toml").spec.output)
CACHE = Path.home() / "goldsluice-output" / "relcal-bstd-82v.npz"

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

if CACHE.exists():
    z = np.load(CACHE)
    series, trigger, bstd = z["series"], z["trigger"], z["bstd"]
else:
    S_, T_, B_ = [], [], []
    for f in sorted((OUT / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
        tab = pq.read_table(f, columns=["series", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        S_.append(np.asarray(tab["series"].to_pylist())); T_.append(np.asarray(tab["trigger_type"].to_pylist()))
        B_.append(np.column_stack([field(tab, k, "baseline_std") for k in LETTERS]))
    series, trigger, bstd = np.concatenate(S_), np.concatenate(T_), np.vstack(B_)
    np.savez(CACHE, series=series, trigger=trigger, bstd=bstd)
real = trigger == "Unknown"
print(f"{len(series):,} events in {len(np.unique(series))} series: {real.sum():,} real triggers, {(trigger == 'Physics').sum():,} random triggers")

## The distribution, channel by channel

Logarithmic horizontal axis: the tail is long. The percentiles (1, 50, 99, 99.9) of the real and of the random triggers are printed below the figure, and the dotted line is 2 times the median of the random triggers.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    r, q = bstd[real, i], bstd[trigger == "Physics", i]
    lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
    bins = np.geomspace(max(lo, 0.3), hi * 1.5, 160)
    ax.hist(q, bins=bins, histtype="stepfilled", color="0.8", density=True, label="random triggers")
    ax.hist(r, bins=bins, histtype="step", color="C0", density=True, lw=1.3, label="real triggers")
    ax.axvline(2 * np.median(q), color="C3", ls=":", lw=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlabel("bstd (counts)")
    ax.set_ylabel("fraction of events per unit bstd")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'':>10}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for name, mask in [("real", real), ("random", trigger == "Physics")]:
    for p in (1, 50, 99, 99.9):
        print(f"{name + ' ' + str(p) + ' %':>10}" + "".join(f"{np.percentile(bstd[mask, i], p):>11.2f}" for i in range(6)))
print(f"{'2 x med.rnd':>10}" + "".join(f"{2 * np.median(bstd[trigger == 'Physics', i]):>11.2f}" for i in range(6)))
print(f"{'real above':>10}" + "".join(f"{(bstd[real, i] > 2 * np.median(bstd[trigger == 'Physics', i])).mean():>10.1%} " for i in range(6)))

## Series by series

The same, one line per series, for the real triggers (each normalised), to see whether the noise or its tail changes along the run. The last row of the printed table is the median of each series.

In [ ]:
sers = sorted(np.unique(series))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cmap = plt.get_cmap("viridis")
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
    bins = np.geomspace(max(lo, 0.3), hi * 1.5, 120)
    for n, s in enumerate(sers):
        m = real & (series == s)
        ax.hist(bstd[m, i], bins=bins, histtype="step", density=True, color=cmap(n / (len(sers) - 1)), lw=1, label=s[4:])
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlabel("bstd (counts)")
axes[0, 0].legend(fontsize=7, title="series (month day _ hour minute)", title_fontsize=7)
fig.tight_layout()
plt.show()

print(f"{'series':>14}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS) + "   median bstd of the real triggers")
for s in sers:
    m = real & (series == s)
    print(f"{s:>14}" + "".join(f"{np.median(bstd[m, i]):>11.2f}" for i in range(6)))

## bstd, height and integral together

The three quantities of a pulse, channel by channel, as the three two-dimensional densities: the height against `bstd`, the integral against `bstd`, and the integral against the height (log colour scale, real triggers only). `height` is `current_max`, the largest sample above the baseline (ADC counts), and `integral` is in ADC counts x s. The series `07221113_1159` (low pulse heights in every channel, see `saturation-over-time.ipynb`) is left out; set `EXCLUDE = set()` to keep it. The dotted lines are twice the median `bstd` of the random triggers, and the saturated height of the channel (the median over the series of the 99.99th percentile of the height).

In [ ]:
EXCLUDE = {"07221113_1159"}
QCACHE = Path.home() / "goldsluice-output" / "relcal-bstd-height-integral-82v.npz"
if QCACHE.exists():
    z = np.load(QCACHE)
    q_series, q_trig, q_integral, q_height = z["series"], z["trigger"], z["integral"], z["height"]
else:
    S_, T_, I_, H_ = [], [], [], []
    for f in sorted((OUT / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
        tab = pq.read_table(f, columns=["series", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        S_.append(np.asarray(tab["series"].to_pylist())); T_.append(np.asarray(tab["trigger_type"].to_pylist()))
        I_.append(np.column_stack([field(tab, k, "integral") for k in LETTERS]))
        H_.append(np.column_stack([field(tab, k, "current_max") for k in LETTERS]))
    q_series, q_trig, q_integral, q_height = np.concatenate(S_), np.concatenate(T_), np.vstack(I_), np.vstack(H_)
    np.savez(QCACHE, series=q_series, trigger=q_trig, integral=q_integral, height=q_height)
assert (q_series == series).all() and (q_trig == trigger).all()          # the same events, in the same order as bstd above
use = real & ~np.isin(series, list(EXCLUDE))
SATH = np.array([1277, 1095, 589, 930, 680, 883.0])                       # saturated height of each channel (saturation-over-time.ipynb)
print(f"{use.sum():,} real triggers")

In [ ]:
from matplotlib import colors

fig, axes = plt.subplots(6, 3, figsize=(16, 24))
for i, k in enumerate(LETTERS):
    b, h, g = bstd[use, i], q_height[use, i], q_integral[use, i]
    noise = 2 * np.median(bstd[trigger == "Physics", i])
    bb = np.geomspace(max(np.percentile(b, 0.01), 0.5), np.percentile(b, 99.99), 110)
    hb = np.geomspace(1, 3000, 110)
    gr = (-0.6, 1.05 * SATH[i] * 2.88e-3)
    gb = np.linspace(*gr, 110)
    for ax, (x, y, xb, yb, xl, yl) in zip(axes[i], [
            (b, h, bb, hb, "bstd (counts)", "height (counts)"),
            (b, g, bb, gb, "bstd (counts)", "integral (counts x s)"),
            (h, g, hb, gb, "height (counts)", "integral (counts x s)")]):
        m = ax.hist2d(x, y, bins=[xb, yb], norm=colors.LogNorm(), cmap="viridis", cmin=1)
        ax.set_xscale("log")
        if yl.startswith("height"):
            ax.set_yscale("log")
        ax.set_xlabel(xl)
        ax.set_ylabel(f"Side2p{k}: {yl}")
    axes[i, 0].axvline(noise, color="C3", ls=":", lw=1)
    axes[i, 0].axhline(SATH[i], color="C3", ls=":", lw=1)
    axes[i, 1].axvline(noise, color="C3", ls=":", lw=1)
    axes[i, 2].axvline(SATH[i], color="C3", ls=":", lw=1)
    axes[i, 0].set_title(f"Side2p{k}  ({use.sum():,} events)", fontsize=10)
fig.tight_layout()
plt.show()

**What this shows, and what is not done.** The baseline noise distribution of every channel, and its relation to the height and the integral. The reason for the ridge (the pulse starts inside the baseline window) is tested in `baseline-window-onset.ipynb`. Not done: a cut. Where to put it (a multiple of the random-trigger median, a percentile, or a limit per series) is the next decision; the dotted line only marks twice the median of the random triggers.